# Driver competition
Compare your driver with `rival_driver.py` on one board or all 14 boards, sequentially. These runs are headless: no simulator window opens.

The rival file is empty initially. Add a class named `IntelligentDriver` (or change `RIVAL_CLASS` below) with the same interface as your driver. Until then, its results say **not_implemented**, and it receives no score.

**Scoring:** boards completed and checkpoints reached measure progress. Checkpoint speed wins compare cumulative arrival times only when both drivers reach that checkpoint on the same board and seed. No speed winner is declared when the rival is missing. Times are simulated seconds, including startup delay; computation time is separate. A timeout keeps any checkpoints already reached. Errors and process timeouts are unscored.

The runner uses the existing car physics, estimator, collisions, and ordered checkpoint checks. It schedules 20 AutoCar updates and 5 traffic/observation updates per simulated second, rather than the GUI's concurrent threads. Random streams for traffic, sensors, and the driver are separate. This is a reproducible comparison, not an exact reproduction of GUI timing.

Run the cells in order. Interrupt the run cell to stop the suite and keep completed results. No simulator or driver implementation files are modified.

In [1]:
from pathlib import Path
from datetime import datetime
import json, sys, html, os
from IPython.display import display, HTML

PROJECT = Path('/Users/aakritypandey/Documents/trying/ai_course/autocar')
PYTHON = '/opt/homebrew/bin/python3.13'
DRIVER_DEBUG = False
os.environ['INTELLIGENT_DRIVER_DEBUG'] = str(DRIVER_DEBUG).lower()
sys.path.insert(0, str(PROJECT))
from driver_competition import run_suite, tables, save_results

RUN_ALL = True                 # True runs every board, one after another.
BOARD = 'small_congested'            # Used when RUN_ALL is False.
SEEDS = [1]                     # Try [1, 2, 3] for a broader comparison.
CARS = None                     # Board defaults: 1 normally, 3 for congested.
PARKED = False
MAX_SIM_SECONDS = 120
PROCESS_TIMEOUT_SECONDS = 60    # Protects against a driver that hangs.
RIVAL_FILE = PROJECT / 'rival_driver.py'
RIVAL_CLASS = 'IntelligentDriver'
DRIVERS = {'Your driver': (PROJECT / 'intelligentDriver.py', 'IntelligentDriver'),
           'Rival': (RIVAL_FILE, RIVAL_CLASS)}

def show_table(rows):
    if not rows:
        print('No results for this table yet.')
        return
    columns = list(rows[0])
    header = ''.join('<th>' + html.escape(c) + '</th>' for c in columns)
    body = ''.join('<tr>' + ''.join('<td>' + html.escape(str(row.get(c, ''))) + '</td>' for c in columns) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + header + '</tr></thead><tbody>' + body + '</tbody></table>'))

boards = json.loads((PROJECT / 'test_boards.json').read_text())
show_table([{'board': name, 'checkpoints': len(b['checkpoints']), 'default_cars': b['default_cars']} for name, b in boards.items()])

board,checkpoints,default_cars
small_local,3,1
small_tour,6,1
small_mixed,5,1
small_mirror_x,4,1
small_mirror_y,5,1
small_rotated,6,1
small_congested,6,3
lombard_local,3,1
lombard_tour,6,1
lombard_mixed,5,1


## Run only your driver
Run the next cell once to define `run_my_driver`, then call it with a board name. Use `run_my_driver()` to run every board. Results are saved under `competition_results`.


In [2]:
def run_my_driver(board=BOARD, seeds=SEEDS, cars=CARS, parked=PARKED,
                  seconds=MAX_SIM_SECONDS, wall_timeout=PROCESS_TIMEOUT_SECONDS):
    """Run intelligentDriver.py alone on one board, or all boards if board=None."""
    import importlib
    import driver_competition as competition_module

    selected = list(boards) if board is None else [board]
    if any(name not in boards for name in selected):
        raise ValueError(f'Unknown board: {board}. Choose from {list(boards)}')
    if seconds <= 0 or wall_timeout <= 0:
        raise ValueError('seconds and wall_timeout must be positive')
    competition_module = importlib.reload(competition_module)
    own_driver = {'Your driver': (PROJECT / 'intelligentDriver.py', 'IntelligentDriver')}
    own_results = competition_module.run_suite(
        selected, own_driver, seeds=seeds, cars=cars, parked=parked,
        seconds=seconds, python=PYTHON, wall_timeout=wall_timeout)
    output = PROJECT / 'competition_results' / ('my_driver_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
    competition_module.save_results(own_results, output)
    show_table([{key: row.get(key) for key in (
        'board', 'seed', 'status', 'checkpoints_reached', 'checkpoints_total',
        'checkpoint_times', 'completion_seconds', 'compute_seconds', 'error')}
        for row in own_results])
    print('Results saved to:', output)
    return own_results

# Example: run_my_driver('small_congested')
# Example: run_my_driver()  # all boards


In [3]:
# Run this cell again after changing the settings above.
# Refresh the runner so edits to driver_competition.py work without restarting Jupyter.
import importlib
import driver_competition as competition_module
competition_module = importlib.reload(competition_module)
run_suite = competition_module.run_suite
tables = competition_module.tables
save_results = competition_module.save_results
selected = list(boards) if RUN_ALL else [BOARD]
assert all(name in boards for name in selected), 'Unknown board name'
assert MAX_SIM_SECONDS > 0 and PROCESS_TIMEOUT_SECONDS > 0

GAME_RUNNING = False
CURRENT_GAME = None

def update_game_status(running, game):
    global GAME_RUNNING, CURRENT_GAME
    GAME_RUNNING = running
    CURRENT_GAME = game if running else None

try:
    results = run_suite(selected, DRIVERS, seeds=SEEDS, cars=CARS, parked=PARKED,
                        seconds=MAX_SIM_SECONDS, python=PYTHON,
                        wall_timeout=PROCESS_TIMEOUT_SECONDS,
                        status_callback=update_game_status)
finally:
    GAME_RUNNING = False
    CURRENT_GAME = None

OUTPUT = PROJECT / 'competition_results' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
save_results(results, OUTPUT)
print('All games finished. GAME_RUNNING =', GAME_RUNNING)
print('Returned', len(results), 'game result(s) in the results variable.')
summary, races = tables(results)
show_table(summary)
show_table(races)
print('Results saved to:', OUTPUT)


small_local | seed 1 | Your driver | running
  still running (1s)
  still running (2s)
  completed: 3/3 checkpoints, 4.9 simulated seconds
  result returned: {'board': 'small_local', 'driver': 'Your driver', 'seed': 1, 'cars': 1, 'parked': False, 'checkpoints_total': 3, 'checkpoints_reached': 3, 'checkpoint_times': [2.8, 3.85, 4.9], 'completion_seconds': 4.9, 'status': 'completed', 'elapsed_seconds': 4.9, 'error': None, 'compute_seconds': 0.9917}
small_local | seed 1 | Rival | running
  still running (1s)
  still running (2s)
  still running (3s)
  still running (4s)
  completed: 3/3 checkpoints, 4.45 simulated seconds
  result returned: {'board': 'small_local', 'driver': 'Rival', 'seed': 1, 'cars': 1, 'parked': False, 'checkpoints_total': 3, 'checkpoints_reached': 3, 'checkpoint_times': [2.35, 3.4, 4.45], 'completion_seconds': 4.45, 'status': 'completed', 'elapsed_seconds': 4.45, 'error': None, 'compute_seconds': 3.7486}
small_tour | seed 1 | Your driver | running
  still running (1s)

driver,scored_runs,boards_completed,checkpoints_reached,checkpoint_speed_wins,mean_completion_seconds,collisions,timeouts,unscored_runs
Rival,14,7,44,18,11.8,7,0,0
Your driver,14,10,56,20,17.215,4,0,0


board,seed,checkpoint,Your driver,Rival,winner
lombard_local,1,1,2.4,2.35,Rival
lombard_local,1,2,3.6,3.3,Rival
lombard_local,1,3,4.65,4.4,Rival
lombard_mirror_x,1,1,3.7,3.45,Rival
lombard_mirror_x,1,2,5.0,5.0,tie
lombard_mirror_x,1,3,6.75,6.8,Your driver
lombard_mirror_x,1,4,9.25,9.7,Your driver
lombard_mirror_y,1,1,5.9,5.9,tie
lombard_mirror_y,1,2,7.7,7.9,Your driver
lombard_mixed,1,1,6.55,6.5,Rival


Results saved to: /Users/aakritypandey/Documents/trying/ai_course/autocar/competition_results/20261114_101758_512146


In [4]:
# Final scoreboard: compare completed boards first, then checkpoint progress.
summary, races = tables(results)
show_table(summary)
if len(summary) == 2 and all(row['scored_runs'] > 0 for row in summary) and summary[0]['scored_runs'] == summary[1]['scored_runs'] and not any(row['unscored_runs'] for row in summary):
    progress = [(r['boards_completed'], r['checkpoints_reached']) for r in summary]
    print('Progress winner:', 'tie' if progress[0] == progress[1] else summary[progress.index(max(progress))]['driver'])
    speed = [r['checkpoint_speed_wins'] for r in summary]
    print('Checkpoint speed winner:', 'no comparable checkpoints' if not races else 'tie' if speed[0] == speed[1] else summary[speed.index(max(speed))]['driver'])
else:
    print('No overall winner: both drivers need complete, comparable sets of scored runs.')

driver,scored_runs,boards_completed,checkpoints_reached,checkpoint_speed_wins,mean_completion_seconds,collisions,timeouts,unscored_runs
Rival,14,7,44,18,11.8,7,0,0
Your driver,14,10,56,20,17.215,4,0,0


Progress winner: Your driver
Checkpoint speed winner: Your driver


In [5]:
# Each checkpoint race shows cumulative arrival seconds from the start.
show_table(races)
# Outcomes for every run, including incomplete and failed runs.
show_table([{key: row.get(key) for key in ['board', 'seed', 'driver', 'status', 'checkpoints_reached', 'checkpoints_total', 'checkpoint_times', 'completion_seconds', 'compute_seconds', 'error']} for row in results])

board,seed,checkpoint,Your driver,Rival,winner
lombard_local,1,1,2.4,2.35,Rival
lombard_local,1,2,3.6,3.3,Rival
lombard_local,1,3,4.65,4.4,Rival
lombard_mirror_x,1,1,3.7,3.45,Rival
lombard_mirror_x,1,2,5.0,5.0,tie
lombard_mirror_x,1,3,6.75,6.8,Your driver
lombard_mirror_x,1,4,9.25,9.7,Your driver
lombard_mirror_y,1,1,5.9,5.9,tie
lombard_mirror_y,1,2,7.7,7.9,Your driver
lombard_mixed,1,1,6.55,6.5,Rival


board,seed,driver,status,checkpoints_reached,checkpoints_total,checkpoint_times,completion_seconds,compute_seconds,error
small_local,1,Your driver,completed,3,3,"[2.8, 3.85, 4.9]",4.9,0.9917,None
small_local,1,Rival,completed,3,3,"[2.35, 3.4, 4.45]",4.45,3.7486,None
small_tour,1,Your driver,completed,6,6,"[5.25, 9.55, 13.25, 17.3, 21.0, 24.5]",24.5,1.0099,None
small_tour,1,Rival,collision,2,6,"[5.55, 9.5]",None,2.3684,None
small_mixed,1,Your driver,completed,5,5,"[2.35, 4.7, 5.3, 6.35, 8.3]",8.3,0.0993,None
small_mixed,1,Rival,completed,5,5,"[1.85, 4.05, 4.65, 6.4, 8.65]",8.65,7.0681,None
small_mirror_x,1,Your driver,completed,4,4,"[3.4, 5.05, 8.85, 11.0]",11.0,1.8543,None
small_mirror_x,1,Rival,completed,4,4,"[3.1, 4.9, 8.7, 11.55]",11.55,5.7395,None
small_mirror_y,1,Your driver,completed,5,5,"[5.15, 8.5, 11.85, 12.65, 15.4]",15.4,0.4266,None
small_mirror_y,1,Rival,completed,5,5,"[5.0, 10.5, 14.4, 15.05, 18.95]",18.95,6.5635,None


## Live competition: choose one board
Run the next cell to open two separate simulator windows, one per driver. Change `LIVE_BOARD` to select a board. Both use the same seed and car count. They race on separate copies of the board, not as two cars on the same board.

The window titles identify the drivers. Games stop at the time limit or finish/crash; press `q` to close each window. Use the results cell below to compare times. Live times are wall-clock seconds and can differ with GUI scheduling; the headless cells above remain the reproducible benchmark.

In [6]:
from pathlib import Path
from datetime import datetime
import json, subprocess

LIVE_PROJECT = Path('/Users/aakritypandey/Documents/trying/ai_course/autocar')
LIVE_PYTHON = '/opt/homebrew/bin/python3.13'
# Board options: checkpoints are (row, col), visited in the listed order.
# Small boards: 12 rows x 16 columns. Lombard boards: 24 rows x 12 columns.
# name                 | checkpoints | default cars | ordered checkpoint cells
# small_local          | 3 | 1 | (2,1), (1,5), (1,10)
# small_tour           | 6 | 1 | (5,14), (6,1), (6,14), (5,1), (7,14), (4,1)
# small_mixed          | 5 | 1 | (5,2), (1,8), (2,10), (4,12), (2,4)
# small_mirror_x       | 4 | 1 | (10,9), (8,2), (2,14), (2,3)
# small_mirror_y       | 5 | 1 | (7,12), (6,2), (3,14), (5,13), (6,4)
# small_rotated        | 6 | 1 | (6,1), (5,14), (6,2), (6,14), (5,1), (7,14)
# small_congested      | 6 | 3 | (5,13), (6,2), (6,13), (5,2), (7,13), (4,2)
# lombard_local        | 3 | 1 | (19,2), (16,4), (16,9)
# lombard_tour         | 6 | 1 | (1,4), (22,10), (2,4), (22,9), (1,5), (22,1)
# lombard_mixed        | 5 | 1 | (9,4), (22,3), (2,4), (5,9), (8,5)
# lombard_mirror_x     | 4 | 1 | (22,1), (20,6), (15,2), (6,4)
# lombard_mirror_y     | 5 | 1 | (12,5), (7,10), (18,10), (5,4), (20,10)
# lombard_rotated      | 6 | 1 | (22,7), (1,1), (22,6), (2,1), (21,7), (1,10)
# lombard_congested    | 6 | 3 | (3,10), (22,9), (2,9), (22,8), (4,10), (21,9)
# mirror_x flips left/right; mirror_y flips top/bottom; rotated turns 180 degrees.
LIVE_BOARD = 'small_congested'
LIVE_SEED = 1
LIVE_CARS = None                 # None uses the board's default
LIVE_PARKED = False
LIVE_SECONDS = 120
LIVE_DRIVERS = {'Your driver': ('intelligentDriver.py', 'IntelligentDriver'),
                'Rival': ('rival_driver.py', 'IntelligentDriver')}

live_boards = json.loads((LIVE_PROJECT / 'test_boards.json').read_text())
assert LIVE_BOARD in live_boards, list(live_boards)
assert LIVE_SECONDS > 0
for old in globals().get('live_games', []):
    if old['process'].poll() is None:
        old['process'].terminate()
        try: old['process'].wait(timeout=3)
        except subprocess.TimeoutExpired: old['process'].kill(); old['process'].wait()
LIVE_OUTPUT = LIVE_PROJECT / 'competition_results' / ('live_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
LIVE_OUTPUT.mkdir(parents=True)
live_games = []
for label, (filename, classname) in LIVE_DRIVERS.items():
    result_file = LIVE_OUTPUT / (Path(filename).stem + '.json')
    log_file = result_file.with_suffix('.log')
    command = [LIVE_PYTHON, '-u', str(LIVE_PROJECT / 'run_test_board.py'), LIVE_BOARD,
               '--driver-file', str(LIVE_PROJECT / filename), '--driver-class', classname,
               '--seed', str(LIVE_SEED), '--seconds', str(LIVE_SECONDS), '--result', str(result_file)]
    if LIVE_CARS is not None: command += ['--cars', str(LIVE_CARS)]
    if LIVE_PARKED: command += ['--parked']
    with log_file.open('w') as log:
        process = subprocess.Popen(command, cwd=LIVE_PROJECT, stdout=log, stderr=subprocess.STDOUT)
    live_games.append({'label': label, 'process': process, 'result': result_file, 'log': log_file})
    print('Launched:', label)
print('Move the windows side by side to watch. Results/logs:', LIVE_OUTPUT)

Launched: Your driver
Launched: Rival
Move the windows side by side to watch. Results/logs: /Users/aakritypandey/Documents/trying/ai_course/autocar/competition_results/live_20261114_101758_631185


In [7]:
# Run again to refresh results after the games finish.
live_results = []
for game in live_games:
    if game['result'].exists():
        row = json.loads(game['result'].read_text())
        row['driver'] = game['label']
        live_results.append(row)
        print(game['label'], row['status'], str(row['checkpoints_reached']) + '/' + str(row['checkpoints_total']), 'checkpoints; arrival seconds:', row['checkpoint_times'])
    elif game['process'].poll() is None:
        print(game['label'], 'is still running.')
    else:
        print(game['label'], 'exited without results:', game['log'].read_text()[-3000:])
if len(live_results) == 2:
    a, b = live_results
    counts = [a['checkpoints_reached'], b['checkpoints_reached']]
    print('Most checkpoints:', 'tie' if counts[0] == counts[1] else live_results[counts.index(max(counts))]['driver'])
    for checkpoint, (ta, tb) in enumerate(zip(a['checkpoint_times'], b['checkpoint_times']), 1):
        winner = 'tie' if abs(ta-tb) < 0.001 else a['driver'] if ta < tb else b['driver']
        print('Checkpoint', checkpoint, 'fastest:', winner, '| seconds:', ta, tb)

Your driver is still running.
Rival is still running.


In [8]:
# Optional: stop/close both live games. Use q in the windows for a recorded quit.
for game in globals().get('live_games', []):
    if game['process'].poll() is None:
        game['process'].terminate()
        try: game['process'].wait(timeout=3)
        except subprocess.TimeoutExpired: game['process'].kill(); game['process'].wait()
print('Live windows stopped.')

Live windows stopped.


## Interpreting the results
- A driver that reaches more checkpoints but completes fewer whole boards has a different tradeoff; both totals are shown.
- Mean completion time includes only completed runs. Do not use it alone to compare drivers that completed different boards.
- Congested boards default to three cars. `CARS` overrides this for every board.
- Every batch saves raw runs to JSON, plus summary and checkpoint-race CSVs where available.
- To watch your driver in a simulator window separately, run `run_test_board.py small_local` from a terminal using the project Python.